# Foundation Models for Forecasting

Companion code for the *Predictive Analytics Lab* (section 28). Run the cell below.

In [ ]:
# Foundation-model building blocks: tokens, patches, quantiles, pinball loss
import numpy as np
AIR = [112,118,132,129,121,135,148,148,136,119,104,118,115,126,141,135,125,149,170,170,158,133,114,140,145,150,178,163,172,178,199,199,184,162,146,166,171,180,193,181,183,218,230,242,209,191,172,194,196,196,236,235,229,243,264,272,237,211,180,201,204,188,235,227,234,264,302,293,259,229,203,229,242,233,267,269,270,315,364,347,312,274,237,278,284,277,317,313,318,374,413,405,355,306,271,306,315,301,356,348,355,422,465,467,404,347,305,336,340,318,362,348,363,435,491,505,404,359,310,337,360,342,406,396,420,472,548,559,463,407,362,405,417,391,419,461,472,535,622,606,508,461,390,432]
y = np.array(AIR, dtype=float); tr, te = y[:120], y[120:]

# 1) Chronos-style value tokens: mean scaling + uniform bins
ctx = tr[-48:]; scaled = ctx / np.mean(np.abs(ctx))
tokens = np.clip(np.digitize(scaled, np.linspace(0.4, 1.6, 33)) - 1, 0, 31)
print("Chronos-style tokens (last 48 months):\n", tokens)

# 2) Patching: 96 values in patches of 8 -> 12 tokens
C, P = 96, 8
patches = tr[-C:].reshape(-1, P)
print(f"\n{C} values -> {len(patches)} patch tokens of length {P}; attention pairs {len(patches)**2} instead of {C**2}")

# 3) A probabilistic baseline: seasonal naive with growth, plus quantiles
yoy = np.log(tr[12:] / tr[:-12]); g, sd = yoy.mean(), yoy.std()
z = {0.1: -1.2816, 0.5: 0.0, 0.9: 1.2816}
h = np.arange(1, 25); k = np.ceil(h / 12)
base = tr[-12:][(h - 1) % 12] * np.exp(g * k)
Q = {q: base * np.exp(zq * sd * np.sqrt(k)) for q, zq in z.items()}

def pinball(y, f, q):
    d = y - f
    return np.mean(np.maximum(q * d, (q - 1) * d))

for q in z:
    print(f"q={q:.1f}: pinball loss = {pinball(te, Q[q], q):6.2f}")
cover = np.mean((te >= Q[0.1]) & (te <= Q[0.9]))
print(f"80% interval coverage on the test set: {cover:.0%};  median RMSE = {np.sqrt(np.mean((Q[0.5] - te) ** 2)):.2f}")

# 4) The real thing: Chronos-2 zero-shot (run in Google Colab after: pip install chronos-forecasting)
try:
    from chronos import BaseChronosPipeline
except ImportError:
    print("\nTo try Chronos-2, open this notebook in Colab, run 'pip install chronos-forecasting', then rerun.")
else:
    import pandas as pd
    df = pd.DataFrame({"id": "air", "timestamp": pd.date_range("1949-01-01", periods=120, freq="MS"), "target": tr})
    pipe = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cpu")
    pred = pipe.predict_df(df, prediction_length=24, quantile_levels=[0.1, 0.5, 0.9],
                           id_column="id", timestamp_column="timestamp", target="target")
    print(pred.head())
    med = pred["predictions"].to_numpy() if "predictions" in pred else pred["0.5"].to_numpy()
    print(f"Chronos-2 zero-shot RMSE: {np.sqrt(np.mean((med - te) ** 2)):.2f}")